# Embedding Responses to Halogen Substitution

## Setup

In [ ]:
from setup import *

from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap, ListedColormap, to_rgb

## Compute Halogen Embeddings

In [ ]:
def halogenate_hexanoic_acid(X, n):
    """Hexanoic acid with n of its 11 C-H replaced by C-X, filled tail first (CH3 end toward the
    carboxyl, like a fluorotelomer). n = 11 is the perhalogenated acid (PFHxA for X = F)."""
    smi, remaining = "", n
    for num_h in [3, 2, 2, 2, 2]:  # C6 (CH3), then C5..C2 (CH2)
        k = min(remaining, num_h)
        remaining -= k
        smi += "C" + f"({X})" * k
    return smi + "C(=O)O"

n_confs = 64
model_paths = {name: CHECKPOINTS[name] for name in ("nopre", "NT", "ANT")}
halogens = ("F", "Cl", "Br")
max_sub = 11
smiles = ["CCCCCC(=O)O"] + [halogenate_hexanoic_acid(X, n) for X in halogens for n in range(1, max_sub + 1)]
halogen_labels = ["parent"] + [f"{X}{n}" for X in halogens for n in range(1, max_sub + 1)]
datalist, mol_idx = generate_conformers(smiles, n_confs, seed=42, max_iters=2000)
pca_coordinates, parent_similarities = {}, {}
for suffix, path in model_paths.items():
    model = EncoderModel.load_model(path, device=DEVICE).eval()
    embeddings = encode_graphs(model, datalist).numpy()
    pca_coordinates[suffix] = compute_pca(embeddings)
    means = np.stack([embeddings[mol_idx == i].mean(0) for i in range(len(halogen_labels))])
    means /= np.linalg.norm(means, axis=1, keepdims=True)
    parent_similarities[suffix] = means @ means[0]

## Figure 4: Halogen Substitution in Hexanoic Acid

In [ ]:
def class_gradient(anchor, n_steps, tint=0.6, shade=0.35):
    """n_steps colors of a single hue, light -> dark (low -> high substitution)."""
    anchor = np.array(to_rgb(anchor))
    ramp = LinearSegmentedColormap.from_list("", [anchor + (1 - anchor) * tint, anchor, anchor * (1 - shade)])
    return [ramp(t) for t in np.linspace(0, 1, n_steps)]

spectral = plt.get_cmap("Spectral")
parent_color = "#4d4d4d"
halogen_fig_size = (5 * cm, 5 * cm)
halogen_anchors = {"F": spectral(0.05), "Cl": spectral(0.78), "Br": spectral(1.0)}
halogen_gradients = {X: class_gradient(anchor, max_sub) for X, anchor in halogen_anchors.items()}
halogen_colors = [parent_color] + [c for X in halogens for c in halogen_gradients[X]]

for model_name, pca in pca_coordinates.items():
    fig = plt.figure(figsize=halogen_fig_size)
    ax = fig.add_axes([0.01, 0.01, 0.98, 0.98])
    for j, X in enumerate(halogens):
        order = [0] + list(range(1 + j * max_sub, 1 + (j + 1) * max_sub))
        centroids = np.stack([pca[mol_idx == i].mean(0) for i in order])
        ax.plot(centroids[:, 0], centroids[:, 1], color=halogen_anchors[X], linewidth=0.5, alpha=0.6, zorder=2)
    for i, color in enumerate(halogen_colors):
        pts = pca[mol_idx == i]
        ax.scatter(pts[:, 0], pts[:, 1], color=color, marker="*" if i == 0 else "o", s=27 if i == 0 else 9,
                   linewidths=0.15, edgecolors="white", zorder=4 if i == 0 else 3)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(0.25)
    fig.savefig(OUTPUT_DIR / f"substitution_halogen_PCA_{model_name}.png", dpi=600)
    plt.show()

height_cm, max_width_cm = 3.0, 2.1
bar_w, bar_gap, pad = 0.25, 0.15, 0.02

def draw_halogen_legend(bar_h):
    fig = plt.figure(figsize=(5 * cm, 5 * cm), dpi=600)
    for j, X in enumerate(halogens):
        cmap = ListedColormap(halogen_gradients[X])
        norm = BoundaryNorm(np.arange(0.5, len(halogen_gradients[X]) + 1), cmap.N)
        cbar = fig.colorbar(matplotlib.cm.ScalarMappable(norm=norm, cmap=cmap),
                           cax=fig.add_axes([(0.5 + j * (bar_w + bar_gap)) / 5, 0.5 / 5, bar_w / 5, bar_h / 5]),
                           orientation="vertical", ticks=[1, 6, 11])
        cbar.ax.set_title(X, fontsize=6, pad=3)
        cbar.ax.tick_params(labelsize=5, length=1.5, width=0.3, pad=1)
        cbar.ax.minorticks_off()
        cbar.outline.set_linewidth(0.3)
        if j < len(halogens) - 1:
            cbar.ax.tick_params(right=False, labelright=False)
    bbox = fig.get_tightbbox(fig.canvas.get_renderer())
    return fig, bbox.width * 2.54 + 2 * pad, bbox.height * 2.54 + 2 * pad

fig, _, trial_h = draw_halogen_legend(height_cm)
plt.close(fig)
fig, width_cm, legend_h = draw_halogen_legend(height_cm - (trial_h - height_cm))
assert width_cm <= max_width_cm and legend_h <= height_cm + 0.01, f"legend is {width_cm:.2f} x {legend_h:.2f} cm"
fig.savefig(OUTPUT_DIR / "substitution_halogen_PCA_legend.png", dpi=600, bbox_inches="tight", pad_inches=pad * cm)
plt.show()

## Cosine Similarity to Hexanoic Acid

In [ ]:
display(pd.DataFrame(parent_similarities, index=halogen_labels).round(3))